In [ ]:
# Default parameter values — overridden by papermill at runtime
run_id = "00000000-0000-0000-0000-000000000000"
bucket = "django-prefect-datalake-dev"
aws_s3_region = "us-east-1"
s3_endpoint = "localhost:9000"
notebook_output_dir = "data/notebook_outputs"
# Scoring algorithm — injected by PipelineRunner from the active ScoringModel.
# Defaults match v1.0 so the notebook works standalone.
scoring_weights = {
    "credit_score": 0.40,
    "income": 0.30,
    "employment_years": 0.20,
    "age": 0.10,
}
scoring_thresholds = {
    "approved": 0.70,
    "review": 0.50,
}


In [ ]:
import json
import s3fs
import polars as pl

In [ ]:
endpoint_url = f"http://{s3_endpoint}" if not s3_endpoint.startswith("http") else s3_endpoint
storage_options = {
    "endpoint_url": endpoint_url,
}
s3 = s3fs.S3FileSystem(
    endpoint_url=endpoint_url,
    client_kwargs={"region_name": aws_s3_region},
)

In [ ]:
input_path = f"s3://{bucket}/processed/flows/credit-prediction/{run_id}/predict_01_raw.parquet"
df = pl.read_parquet(input_path, storage_options=storage_options)
row = df.row(0, named=True)

In [ ]:
income = float(row["income"])
age = float(row["age"])
credit_score = float(row["credit_score"])
employment_years = float(row["employment_years"])

# Normalise each factor to [0, 1]
credit_score_norm = max(0.0, min(1.0, (credit_score - 300) / 550))  # FICO 300-850
income_norm       = max(0.0, min(1.0, income / 150_000))             # up to $150k
employment_norm   = max(0.0, min(1.0, employment_years / 20))        # up to 20 yrs
age_norm          = 1.0 if 22 <= age <= 70 else 0.6                  # optimal age band

# Weighted score — weights sourced from ScoringModel (injected by PipelineRunner)
score = (
    credit_score_norm * scoring_weights["credit_score"]
    + income_norm       * scoring_weights["income"]
    + employment_norm   * scoring_weights["employment_years"]
    + age_norm          * scoring_weights["age"]
)
score = round(score, 4)
confidence = round(score * 100, 1)

# Classification thresholds sourced from ScoringModel
if score >= scoring_thresholds["approved"]:
    classification = "Approved"
elif score >= scoring_thresholds["review"]:
    classification = "Review"
else:
    classification = "Declined"

print(f"Score: {score}  Classification: {classification}  Confidence: {confidence}%")


In [ ]:
result_df = pl.DataFrame({
    "income":           [income],
    "age":              [age],
    "credit_score":     [credit_score],
    "employment_years": [employment_years],
    "score":            [score],
    "classification":   [classification],
    "confidence":       [confidence],
})

s3_output_path = f"processed/flows/credit-prediction/{run_id}/output.parquet"
output_full = f"s3://{bucket}/{s3_output_path}"

with s3.open(output_full, "wb") as f:
    result_df.write_parquet(f, compression="snappy", use_pyarrow=True)

print(f"Scored → {output_full}")

In [ ]:
result = {
    "row_count": 1,
    "s3_output_path": s3_output_path,
    "score": score,
    "classification": classification,
    "confidence": confidence,
}

# Write result.json to S3 — primary result contract (BL-029)
result_path = f"processed/flows/credit-prediction/{run_id}/result.json"
with s3.open(f"s3://{bucket}/{result_path}", "w") as f:
    json.dump(result, f)

# Keep stdout print for backwards compatibility and local debugging
print(json.dumps(result))
